# Experiment EDA

Один notebook для всіх експериментів. Він читає той самий `experiment.toml`, що й команди пайплайна, та будує графіки з версованого `report/data.json`.

In [ ]:
import json
import os
import tomllib
from html import escape
from pathlib import Path
from statistics import fmean

from IPython.display import HTML, display

experiment_path = (
    Path(os.environ.get("EXPERIMENT_CONFIG", "experiment.toml")).expanduser().resolve()
)
experiment = tomllib.loads(experiment_path.read_text(encoding="utf-8"))
required_keys = {"schema_version", "experiment_id", "experiment_version", "artifact_dir"}
if experiment.get("schema_version") != 1 or not required_keys.issubset(experiment):
    raise ValueError(f"invalid experiment config: {experiment_path}")
artifact_dir = Path(experiment["artifact_dir"]).expanduser()
if not artifact_dir.is_absolute():
    artifact_dir = experiment_path.parent / artifact_dir
artifact_root = (
    artifact_dir / experiment["experiment_id"] / experiment["experiment_version"]
).resolve()
data_path = artifact_root / "report/data.json"
data = json.loads(data_path.read_text(encoding="utf-8"))
for key in ("experiment_id", "experiment_version"):
    if data.get(key) != experiment[key]:
        raise ValueError(f"EDA data has the wrong {key}")

rows = data["rows"]
conditions = [row["condition"] for row in data["aggregates"]]
aggregates = {row["condition"]: row for row in data["aggregates"]}
if len(aggregates) != len(conditions):
    raise ValueError("EDA aggregates contain duplicate conditions")
display(
    HTML(
        f"<h2>{escape(data['experiment_id'])} · {escape(data['experiment_version'])}</h2>"
        f"<p><code>{escape(str(data_path))}</code><br>"
        f"plan <code>{escape(data['plan_id'])}</code></p>"
    )
)

In [ ]:
COLORS = {"NO-DOC": "#64748b", "OPTIONAL": "#f59e0b", "DOC-FIRST": "#2563eb"}


def metric(value, digits=3):
    return "—" if value is None else f"{value:.{digits}f}"


def bars(title, values, *, digits=3):
    available = [value for _, value in values if value is not None]
    maximum = max(available, default=1) or 1
    body = []
    for label, value in values:
        width = 0 if value is None else 100 * value / maximum
        color = COLORS.get(label, "#0f766e")
        body.append(
            "<div style='display:grid;grid-template-columns:100px 1fr 90px;"
            "gap:12px;align-items:center;margin:8px 0'>"
            f"<strong>{escape(label)}</strong><div style='background:#e8edf4;border-radius:5px'>"
            f"<div style='width:{width:.2f}%;height:22px;"
            f"background:{color};border-radius:5px'></div></div>"
            f"<code>{metric(value, digits)}</code></div>"
        )
    display(HTML(f"<h3>{escape(title)}</h3>{''.join(body)}"))

## Якість локалізації

In [ ]:
quality_metrics = {
    "Recall@3": "mean_recall_at_3",
    "nDCG@3": "mean_ndcg_at_3",
    "Returned-set F1": "mean_returned_set_f1",
    "Recall@5": "mean_recall_at_5",
}
for title, key in quality_metrics.items():
    bars(title, [(condition, aggregates[condition][key]) for condition in conditions])

## Фічі тексту задачі

Публічні locator-like фічі виділяються з prompt на `features`; відповідність gold-файлу або Python-символу додається лише на `analyze`.

In [ ]:
TASK_FEATURES = {
    "prompt_has_path": "Path у prompt",
    "prompt_has_filename": "Filename у prompt",
    "prompt_has_symbol": "Symbol у prompt",
    "gold_locator_mentioned": "Підтверджена gold-підказка",
}
task_features = {}
for row in rows:
    current = {key: row[key] for key in TASK_FEATURES if key in row}
    previous = task_features.setdefault(row["task_id"], current)
    if previous != current:
        raise ValueError(f"inconsistent task features: {row['task_id']}")
available_features = [
    key for key in TASK_FEATURES if all(key in row for row in task_features.values())
]
feature_rows = []
for key in available_features:
    present = sum(row[key] for row in task_features.values())
    feature_rows.append(
        f"<tr><th>{escape(TASK_FEATURES[key])}</th><td>{present}</td>"
        f"<td>{len(task_features) - present}</td></tr>"
    )
display(
    HTML(
        "<table><thead><tr><th>Фіча</th><th>Є</th><th>Немає</th></tr></thead><tbody>"
        + "".join(feature_rows)
        + "</tbody></table>"
    )
)
if "gold_locator_mentioned" in available_features:
    locator_quality = []
    for label, value in (("Немає", False), ("Є", True)):
        selected = [
            row["recall_at_3"]
            for row in rows
            if row["status"] == "succeeded"
            and row["gold_locator_mentioned"] is value
            and row["recall_at_3"] is not None
        ]
        locator_quality.append((label, fmean(selected) if selected else None))
    bars("Recall@3 за підтвердженою gold-підказкою", locator_quality)

## Вартість виконання

In [ ]:
successful = [row for row in rows if row["status"] == "succeeded"]


def condition_mean(condition, value):
    values = [value(row) for row in successful if row["condition"] == condition]
    values = [item for item in values if item is not None]
    return fmean(values) if values else None


bars(
    "Середній час, секунд",
    [
        (condition, condition_mean(condition, lambda row: row["duration_ms"] / 1000))
        for condition in conditions
    ],
    digits=1,
)
bars(
    "Середні input + output tokens",
    [
        (
            condition,
            condition_mean(
                condition,
                lambda row: None
                if row["input_tokens"] is None or row["output_tokens"] is None
                else row["input_tokens"] + row["output_tokens"],
            ),
        )
        for condition in conditions
    ],
    digits=0,
)
bars(
    "Середні tool steps",
    [
        (condition, condition_mean(condition, lambda row: row["tool_steps"]))
        for condition in conditions
    ],
    digits=1,
)

## Розподіл за репозиторіями

In [ ]:
repositories = sorted({row["repository"] for row in rows})
header = "".join(f"<th>{escape(condition)}</th>" for condition in conditions)
table_rows = []
for repository in repositories:
    cells = []
    for condition in conditions:
        values = [
            row["recall_at_3"]
            for row in successful
            if row["repository"] == repository
            and row["condition"] == condition
            and row["recall_at_3"] is not None
        ]
        cells.append(f"<td>{metric(fmean(values) if values else None)}</td>")
    table_rows.append(f"<tr><th>{escape(repository)}</th>{''.join(cells)}</tr>")
display(
    HTML(
        "<table><caption>Mean Recall@3</caption><thead><tr><th>Repository</th>"
        + header
        + "</tr></thead><tbody>"
        + "".join(table_rows)
        + "</tbody></table>"
    )
)